Revenue Leakage Analysis in a Subscription Learning Platform (SQL Case Study)

### Business Problem Statement
A subscription-based learning platform has noticed that even though new users are joining every week, monthly revenue is not growing as expected.

Your task is to analyze subscription behavior and identify where revenue leakage is happening.

### Questions to Solve Using SQL:
1. Find the total number of users in the platform.
2. Count how many users signed up in each country.
3. Find how many users signed up each month.
4. Find how many users are currently active vs cancelled.
5. Find how many users subscribed to each plan.
6. Find how many users cancelled their subscription within 30 days of starting.
7. Find which subscription plan has the highest number of cancellations.
8. Find the number of failed payments grouped by payment method.
9. Find the total revenue successfully collected per payment method.
10. Find how many users downgraded their subscription plans.
11. Calculate the percentage of users who downgraded.
12. Find total revenue generated by each subscription plan.
13. Find total revenue generated per country.
14. Find top 5 highest-paying users based on total payments made.
15. Segment users into Low, Medium, and High value based on total revenue generated.
16. Which combination of plan_name and country generates the highest revenue?

In [41]:
import pandas as pd
import sqlite3

excel_file = '/content/subscription_revenue_leakage_dataset.xlsx'

# Load each relevant sheet into its own DataFrame
users_df = pd.read_excel(excel_file, sheet_name='users', header=0)
plans_df = pd.read_excel(excel_file, sheet_name='plans', header=0)
subscriptions_df = pd.read_excel(excel_file, sheet_name='subscriptions', header=0)
payments_df = pd.read_excel(excel_file, sheet_name='payments', header=0)
plan_changes_df = pd.read_excel(excel_file, sheet_name='plan_changes', header=0)

print("DataFrames loaded successfully. Displaying head and info for each:")

print("\n--- users_df ---")
display(users_df.head())
users_df.info()

print("\n--- plans_df ---")
display(plans_df.head())
plans_df.info()

print("\n--- subscriptions_df ---")
display(subscriptions_df.head())
subscriptions_df.info()
print("Subscriptions DataFrame columns:", subscriptions_df.columns.tolist())

print("\n--- payments_df ---")
display(payments_df.head())
payments_df.info()
print("Payments DataFrame columns:", payments_df.columns.tolist())

print("\n--- plan_changes_df ---")
display(plan_changes_df.head())
plan_changes_df.info()

# Note: 'README', 'data_dictionary', and 'relationships' sheets are not loaded as primary data tables for analysis,
# but can be inspected for schema understanding if needed.

DataFrames loaded successfully. Displaying head and info for each:

--- users_df ---


,user_sk,user_id,sign_up_date,country,age_group,occupation_segment,acquisition_channel,primary_device,billing_country,referred_by_friend,student_discount_eligible
0,1,U00001,2024-04-12,Canada,35-44,Student,YouTube,Mobile,Canada,Yes,Yes
1,2,U00002,2024-09-27,Australia,18-24,Career Switcher,Organic Search,Mobile,Australia,No,No
2,3,U00003,2025-12-01,United Kingdom,25-34,Student,Organic Search,Desktop,United Kingdom,No,Yes
3,4,U00004,2024-05-01,United States,35-44,Working Professional,Email,Mobile,United States,No,No
4,5,U00005,2024-11-26,India,18-24,Student,Affiliate,Mobile,India,No,Yes


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   user_sk                    2500 non-null   int64         
 1   user_id                    2500 non-null   object        
 2   sign_up_date               2500 non-null   datetime64[ns]
 3   country                    2500 non-null   object        
 4   age_group                  2500 non-null   object        
 5   occupation_segment         2500 non-null   object        
 6   acquisition_channel        2500 non-null   object        
 7   primary_device             2500 non-null   object        
 8   billing_country            2500 non-null   object        
 9   referred_by_friend         2500 non-null   object        
 10  student_discount_eligible  2500 non-null   object        
dtypes: datetime64[ns](1), int64(1), object(9)
memory usage: 215.0+ KB

--

,plan_id,plan_name,plan_tier,billing_cycle,price_usd,trial_days,billing_interval_months,plan_description
0,1,Basic Monthly,Basic,Monthly,19,7,1,Entry plan with course access and community su...
1,2,Pro Monthly,Pro,Monthly,39,14,1,"Includes certificates, assessments, and live d..."
2,3,Premium Monthly,Premium,Monthly,79,14,1,"Includes mentorship, projects, and premium sup..."
3,4,Pro Annual,Pro,Annual,399,14,12,Discounted annual version of Pro
4,5,Premium Annual,Premium,Annual,799,14,12,Discounted annual version of Premium


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   plan_id                  5 non-null      int64 
 1   plan_name                5 non-null      object
 2   plan_tier                5 non-null      object
 3   billing_cycle            5 non-null      object
 4   price_usd                5 non-null      int64 
 5   trial_days               5 non-null      int64 
 6   billing_interval_months  5 non-null      int64 
 7   plan_description         5 non-null      object
dtypes: int64(4), object(4)
memory usage: 452.0+ bytes

--- subscriptions_df ---


...

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   subscription_sk         2500 non-null   int64         
 1   subscription_id         2500 non-null   object        
 2   user_id                 2500 non-null   object        
 3   initial_plan_id         2500 non-null   int64         
 4   current_plan_id         2500 non-null   int64         
 5   trial_start_date        2500 non-null   datetime64[ns]
 6   trial_end_date          2500 non-null   datetime64[ns]
 7   activation_date         2500 non-null   datetime64[ns]
 8   subscription_status     2500 non-null   object        
 9   cancel_date             807 non-null    datetime64[ns]
 10  cancel_reason           946 non-null    object        
 11  auto_renew_enabled      2500 non-null   object        
 12  default_payment_method  2500 non-null   object  

,payment_sk,payment_id,subscription_id,user_id,payment_date,billing_period_start,billing_period_end,amount_usd,payment_method,payment_type,payment_status,failure_reason,invoice_id
0,1,P000001,S00001,U00001,2024-04-26,2024-04-26,2024-05-25,39.0,UPI,initial,success,NaN,INV000001
1,2,P000002,S00001,U00001,2024-05-26,2024-05-26,2024-06-25,39.0,UPI,renewal,success,NaN,INV000002
2,3,P000003,S00001,U00001,2024-06-26,2024-06-26,2024-07-25,39.0,UPI,renewal,success,NaN,INV000003
3,4,P000004,S00001,U00001,2024-07-26,2024-07-26,2024-08-25,39.0,UPI,renewal,success,NaN,INV000004
4,5,P000005,S00001,U00001,2024-08-26,2024-08-26,2024-09-25,39.0,UPI,renewal,success,NaN,INV000005


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20514 entries, 0 to 20513
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   payment_sk            20514 non-null  int64         
 1   payment_id            20514 non-null  object        
 2   subscription_id       20514 non-null  object        
 3   user_id               20514 non-null  object        
 4   payment_date          20514 non-null  datetime64[ns]
 5   billing_period_start  20514 non-null  datetime64[ns]
 6   billing_period_end    20514 non-null  datetime64[ns]
 7   amount_usd            20514 non-null  float64       
 8   payment_method        20514 non-null  object        
 9   payment_type          20514 non-null  object        
 10  payment_status        20514 non-null  object        
 11  failure_reason        2037 non-null   object        
 12  invoice_id            20514 non-null  object        
dtypes: datetime64[ns

,plan_change_sk,plan_change_id,subscription_id,user_id,old_plan_id,new_plan_id,change_date,change_type,effective_next_cycle
0,1,PC00001,S00008,U00008,3,2,2025-02-20,downgrade,Yes
1,2,PC00002,S00008,U00008,2,1,2025-06-18,downgrade,No
2,3,PC00003,S00010,U00010,1,2,2025-03-08,upgrade,Yes
3,4,PC00004,S00014,U00014,2,3,2026-01-17,upgrade,Yes
4,5,PC00005,S00017,U00017,2,3,2025-12-09,upgrade,No


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 331 entries, 0 to 330
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   plan_change_sk        331 non-null    int64         
 1   plan_change_id        331 non-null    object        
 2   subscription_id       331 non-null    object        
 3   user_id               331 non-null    object        
 4   old_plan_id           331 non-null    int64         
 5   new_plan_id           331 non-null    int64         
 6   change_date           331 non-null    datetime64[ns]
 7   change_type           331 non-null    object        
 8   effective_next_cycle  331 non-null    object        
dtypes: datetime64[ns](1), int64(3), object(5)
memory usage: 23.4+ KB


In [24]:
# Create an in-memory SQLite database connection
conn = sqlite3.connect(':memory:')

# Write each DataFrame to a separate SQL table
users_df.to_sql('users', conn, if_exists='replace', index=False)
plans_df.to_sql('plans', conn, if_exists='replace', index=False)
subscriptions_df.to_sql('subscriptions', conn, if_exists='replace', index=False)
payments_df.to_sql('payments', conn, if_exists='replace', index=False)
plan_changes_df.to_sql('plan_changes', conn, if_exists='replace', index=False)

print("All DataFrames loaded into SQLite tables successfully.")

# Verify table creation
print("\nTables in the database:")
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
for table in tables:
    print(table[0])

All DataFrames loaded into SQLite tables successfully.

Tables in the database:
users
plans
subscriptions
payments
plan_changes


In [61]:
print("\n--- SQLite Table Schemas ---")

def get_table_columns(conn, table_name):
    cursor = conn.cursor()
    cursor.execute(f"PRAGMA table_info({table_name});")
    columns = [col[1] for col in cursor.fetchall()]
    return columns

subscriptions_cols = get_table_columns(conn, 'subscriptions')
payments_cols = get_table_columns(conn, 'payments')

print(f"Subscriptions table columns: {subscriptions_cols}")
print(f"Payments table columns: {payments_cols}")


--- SQLite Table Schemas ---
Subscriptions table columns: ['subscription_sk', 'subscription_id', 'user_id', 'initial_plan_id', 'current_plan_id', 'trial_start_date', 'trial_end_date', 'activation_date', 'subscription_status', 'cancel_date', 'cancel_reason', 'auto_renew_enabled', 'default_payment_method', 'coupon_code']
Payments table columns: ['payment_sk', 'payment_id', 'subscription_id', 'user_id', 'payment_date', 'billing_period_start', 'billing_period_end', 'amount_usd', 'payment_method', 'payment_type', 'payment_status', 'failure_reason', 'invoice_id']


In [60]:
print("\n--- SQLite Table Schemas ---")

def get_table_columns(conn, table_name):
    cursor = conn.cursor()
    cursor.execute(f"PRAGMA table_info({table_name});")
    columns = [col[1] for col in cursor.fetchall()]
    return columns

subscriptions_cols = get_table_columns(conn, 'subscriptions')
payments_cols = get_table_columns(conn, 'payments')

print(f"Subscriptions table columns: {subscriptions_cols}")
print(f"Payments table columns: {payments_cols}")


--- SQLite Table Schemas ---
Subscriptions table columns: ['subscription_sk', 'subscription_id', 'user_id', 'initial_plan_id', 'current_plan_id', 'trial_start_date', 'trial_end_date', 'activation_date', 'subscription_status', 'cancel_date', 'cancel_reason', 'auto_renew_enabled', 'default_payment_method', 'coupon_code']
Payments table columns: ['payment_sk', 'payment_id', 'subscription_id', 'user_id', 'payment_date', 'billing_period_start', 'billing_period_end', 'amount_usd', 'payment_method', 'payment_type', 'payment_status', 'failure_reason', 'invoice_id']


The data has been successfully loaded into an in-memory SQLite database table named `subscriptions`. We can now proceed with answering the SQL questions.

### Question 1: Find the total number of users in the platform.

In [25]:
query_q1 = """
SELECT COUNT(DISTINCT user_id) AS total_users
FROM users;
"""

q1_result = pd.read_sql_query(query_q1, conn)
display(q1_result)

,total_users
0,2500


### Question 2: Count how many users signed up in each country.

In [26]:
query_q2 = """
SELECT country, COUNT(DISTINCT user_id) AS users_count
FROM users
GROUP BY country
ORDER BY users_count DESC;
"""

q2_result = pd.read_sql_query(query_q2, conn)
display(q2_result)

,country,users_count
0,United States,602
1,India,555
2,Nigeria,337
3,United Kingdom,262
4,Canada,229
5,Germany,196
6,UAE,166
7,Australia,153


### Question 3: Find how many users signed up each month.

In [27]:
query_q3 = """
SELECT
    STRFTIME('%Y-%m', sign_up_date) AS signup_month,
    COUNT(DISTINCT user_id) AS monthly_signups
FROM users
GROUP BY signup_month
ORDER BY signup_month;
"""

q3_result = pd.read_sql_query(query_q3, conn)
display(q3_result)

,signup_month,monthly_signups
0,2024-01,95
1,2024-02,92
2,2024-03,104
3,2024-04,118
4,2024-05,112
5,2024-06,96
6,2024-07,116
7,2024-08,124
8,2024-09,105
9,2024-10,113


### Question 4: Find how many users are currently active vs cancelled.

In [62]:
query_q4 = """
SELECT
    CASE
        WHEN cancel_date IS NULL THEN 'Active'
        ELSE 'Cancelled'
    END AS status,
    COUNT(DISTINCT user_id) AS user_count
FROM subscriptions
GROUP BY status;
"""

q4_result = pd.read_sql_query(query_q4, conn)
display(q4_result)

,status,user_count
0,Active,1693
1,Cancelled,807


### Question 5: Find how many users subscribed to each plan.

In [29]:
query_q5 = """
SELECT
    p.plan_name,
    COUNT(DISTINCT s.user_id) AS subscribers_count
FROM subscriptions s
JOIN plans p ON s.initial_plan_id = p.plan_id
GROUP BY p.plan_name
ORDER BY subscribers_count DESC;
"""

q5_result = pd.read_sql_query(query_q5, conn)
display(q5_result)

,plan_name,subscribers_count
0,Pro Monthly,807
1,Basic Monthly,656
2,Pro Annual,469
3,Premium Monthly,339
4,Premium Annual,229


### Question 6: Find how many users cancelled their subscription within 30 days of starting.

In [63]:
query_q6 = """
SELECT COUNT(DISTINCT user_id) AS early_cancellations
FROM subscriptions
WHERE cancel_date IS NOT NULL
  AND (JULIANDAY(cancel_date) - JULIANDAY(activation_date)) <= 30;
"""

q6_result = pd.read_sql_query(query_q6, conn)
display(q6_result)

,early_cancellations
0,206


### Question 7: Find which subscription plan has the highest number of cancellations.

In [64]:
query_q7 = """
SELECT
    p.plan_name,
    COUNT(DISTINCT s.user_id) AS cancelled_users
FROM subscriptions s
JOIN plans p ON s.initial_plan_id = p.plan_id
WHERE s.cancel_date IS NOT NULL
GROUP BY p.plan_name
ORDER BY cancelled_users DESC
LIMIT 1;
"""

q7_result = pd.read_sql_query(query_q7, conn)
display(q7_result)

,plan_name,cancelled_users
0,Pro Monthly,318


### Question 8: Find the number of failed payments grouped by payment method.

In [32]:
query_q8 = """
SELECT
    payment_method,
    COUNT(payment_id) AS failed_payments
FROM payments
WHERE payment_status = 'failed'
GROUP BY payment_method
ORDER BY failed_payments DESC;
"""

q8_result = pd.read_sql_query(query_q8, conn)
display(q8_result)

,payment_method,failed_payments
0,Credit Card,738
1,UPI,626
2,Debit Card,412
3,PayPal,151
4,Bank Transfer,110


### Question 9: Find the total revenue successfully collected per payment method.

In [73]:
query_q9 = """
SELECT
    payment_method,
    SUM(amount_usd) AS total_collected_revenue
FROM payments
WHERE payment_status = 'success'
GROUP BY payment_method
ORDER BY total_collected_revenue DESC;
"""

q9_result = pd.read_sql_query(query_q9, conn)
display(q9_result)

,payment_method,total_collected_revenue
0,Credit Card,480587.00
1,UPI,261385.85
2,Debit Card,207782.70
3,PayPal,162656.90
4,Bank Transfer,117331.65


### Question 10: Find how many users downgraded their subscription plans.

In [34]:
query_q10 = """
SELECT COUNT(DISTINCT s.user_id) AS downgraded_users
FROM subscriptions s
JOIN plan_changes pc ON s.subscription_id = pc.subscription_id
WHERE pc.change_type = 'downgrade';
"""

q10_result = pd.read_sql_query(query_q10, conn)
display(q10_result)

,downgraded_users
0,112


### Question 11: Calculate the percentage of users who downgraded.

In [35]:
query_q11 = """
WITH DowngradedUsers AS (
    SELECT COUNT(DISTINCT s.user_id) AS num_downgraded
    FROM subscriptions s
    JOIN plan_changes pc ON s.subscription_id = pc.subscription_id
    WHERE pc.change_type = 'downgrade'
),
TotalUsers AS (
    SELECT COUNT(DISTINCT user_id) AS num_total
    FROM users
)
SELECT
    CAST(num_downgraded AS REAL) * 100 / num_total AS percentage_downgraded
FROM DowngradedUsers, TotalUsers;
"""

q11_result = pd.read_sql_query(query_q11, conn)
display(q11_result)

,percentage_downgraded
0,4.48


### Question 12: Find total revenue generated by each subscription plan.

In [74]:
query_q12 = """
SELECT
    p.plan_name,
    SUM(pm.amount_usd) AS total_revenue
FROM payments pm
JOIN subscriptions s ON pm.subscription_id = s.subscription_id
JOIN plans p ON s.initial_plan_id = p.plan_id
WHERE pm.payment_status = 'success'
GROUP BY p.plan_name
ORDER BY total_revenue DESC;
"""

q12_result = pd.read_sql_query(query_q12, conn)
display(q12_result)

,plan_name,total_revenue
0,Pro Monthly,308012.60
1,Premium Monthly,280899.30
2,Pro Annual,263711.85
3,Premium Annual,252663.35
4,Basic Monthly,124457.00


### Question 13: Find total revenue generated per country.

In [75]:
query_q13 = """
SELECT
    u.country,
    SUM(pm.amount_usd) AS total_revenue
FROM payments pm
JOIN subscriptions s ON pm.subscription_id = s.subscription_id
JOIN users u ON s.user_id = u.user_id
WHERE pm.payment_status = 'success'
GROUP BY u.country
ORDER BY total_revenue DESC;
"""

q13_result = pd.read_sql_query(query_q13, conn)
display(q13_result)

,country,total_revenue
0,United States,295223.20
1,India,276374.60
2,Nigeria,160825.20
3,United Kingdom,132029.60
4,Canada,123140.70
5,Germany,95247.70
6,UAE,74359.15
7,Australia,72543.95


### Question 14: Find top 5 highest-paying users based on total payments made.

In [76]:
query_q14 = """
SELECT
    user_id,
    SUM(amount_usd) AS total_payments
FROM payments
WHERE payment_status = 'success'
GROUP BY user_id
ORDER BY total_payments DESC
LIMIT 5;
"""

q14_result = pd.read_sql_query(query_q14, conn)
display(q14_result)

,user_id,total_payments
0,U02042,2388.0
1,U01882,1975.0
2,U01616,1975.0
3,U01606,1975.0
4,U00766,1975.0


### Question 15: Segment users into Low, Medium, and High value based on total revenue generated.

In [77]:
query_q15 = """
WITH UserTotalRevenue AS (
    SELECT
        user_id,
        SUM(amount_usd) AS total_revenue
    FROM payments
    WHERE payment_status = 'success'
    GROUP BY user_id
),
RankedUsers AS (
    SELECT
        user_id,
        total_revenue,
        NTILE(3) OVER (ORDER BY total_revenue) AS revenue_tier
    FROM UserTotalRevenue
)
SELECT
    CASE
        WHEN revenue_tier = 1 THEN 'Low Value'
        WHEN revenue_tier = 2 THEN 'Medium Value'
        WHEN revenue_tier = 3 THEN 'High Value'
        ELSE 'Unknown'
    END AS user_segment,
    COUNT(user_id) AS user_count
FROM RankedUsers
GROUP BY user_segment
ORDER BY user_count DESC;
"""

q15_result = pd.read_sql_query(query_q15, conn)
display(q15_result)

,user_segment,user_count
0,Low Value,834
1,Medium Value,833
2,High Value,833


### Question 16: Which combination of plan_name and country generates the highest revenue?

In [78]:
query_q16 = """
SELECT
    p.plan_name,
    u.country,
    SUM(pm.amount_usd) AS total_revenue
FROM payments pm
JOIN subscriptions s ON pm.subscription_id = s.subscription_id
JOIN plans p ON s.initial_plan_id = p.plan_id
JOIN users u ON s.user_id = u.user_id
WHERE pm.payment_status = 'success'
GROUP BY p.plan_name, u.country
ORDER BY total_revenue DESC
LIMIT 1;
"""

q16_result = pd.read_sql_query(query_q16, conn)
display(q16_result)

,plan_name,country,total_revenue
0,Pro Annual,United States,78014.25


In [72]:
query_check_payments_count = """
SELECT COUNT(*) AS total_payments,
       SUM(CASE WHEN payment_status = 'successful' THEN 1 ELSE 0 END) AS successful_payments_count
FROM payments;
"""

check_payments_count_result = pd.read_sql_query(query_check_payments_count, conn)
display(check_payments_count_result)

,total_payments,successful_payments_count
0,20514,0


In [71]:
query_check_payment_status = """
SELECT DISTINCT payment_status, COUNT(*) as count
FROM payments
GROUP BY payment_status;
"""

check_payment_status_result = pd.read_sql_query(query_check_payment_status, conn)
display(check_payment_status_result)

,payment_status,count
0,failed,2037
1,success,18477
